# NB1 — Aquisição, Validação e Pré-processamento de Contextos de Crise

**Pipeline de Predição de Crises Epilépticas a partir de EEG**

Este notebook:
1. Define os contextos de crise válidos para os 24 pacientes selecionados
2. Verifica integridade de cada contexto
3. Aplica filtros (passa-alta 0.5Hz, passa-baixa 40Hz, notch 50/60Hz) e reamostra para 256Hz
4. Salva os sinais pré-processados em `.npz` por contexto
5. Exibe tabela detalhada de todos os pacientes e contextos
6. Plota o EEG de cada contexto válido
7. Exporta o mapa de contextos em JSON para o NB2

**Parâmetros da lógica de contexto:**
- `GUARD = 30 min` — zona proibida pós-crise e margem antes do pré-ictal
- `MAX_PRE = 40 min` — janela pré-ictal máxima
- `MIN_INTER = 40 min` — interictal mínimo (simétrico ao pré-ictal)
- `MIN_CONTEXTS = 2` — contextos válidos mínimos para o paciente ser selecionável

**Estrutura do contexto:** `[interictal 40min][guard 30min][pré-ictal 40min][CRISE][guard 30min]`


## 1. Imports e parâmetros

In [2]:
import os, re, json, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore', category=RuntimeWarning)

try:
    import mne
    mne.set_log_level('ERROR')
    HAS_MNE = True
except ImportError:
    HAS_MNE = False
    print('MNE nao instalado — instale com: pip install mne')

# ── Diretorios ────────────────────────────────────────────────────────────────
ROOT            = Path('data')
CHBMIT_DIR      = ROOT / 'chb-mit'
SIENA_DIR       = ROOT / 'siena'
SEIZEIT_DIR     = ROOT / 'seizeit2'
MENDELEY_DIR    = ROOT / 'mendeley'
SIGNAL_DIR      = ROOT / 'signals'    # .npz dos contextos pre-processados
SEIZEIT_SESSION = 'ses-01'
SIGNAL_DIR.mkdir(parents=True, exist_ok=True)

# ── Parâmetros do contexto de crise ──────────────────────────────────────────
GUARD_S     = 30 * 60   # 30 min
MAX_PRE_S   = 40 * 60   # 40 min
MIN_INTER_S = 40 * 60   # 40 min
MIN_CONTEXTS = 2

# ── Parâmetros de pré-processamento ──────────────────────────────────────────
TARGET_SFREQ = 256.0    # Hz — reamostragem alvo
F_HIGHPASS   = 0.5      # Hz — passa-alta (remove drift DC)
F_LOWPASS    = 40.0     # Hz — passa-baixa (remove ruido alta frequencia)
NOTCH_FREQS  = [50, 60] # Hz — notch (rede eletrica Europa + EUA)

# ── Seleção automática de pacientes ──────────────────────────────────────────
# Os pacientes são descobertos automaticamente em disco e rankeados por número
# de contextos válidos. São selecionados TODOS com >= MIN_CONTEXTS contextos.
# Não há lista manual — a seleção é determinada pelos parâmetros e pelos dados.
#
# N_SELECT_PER_DS: None = todos os elegíveis; int = top-N por dataset.
# Recomendado: None (seleção data-driven, sem viés de curadoria manual).
N_SELECT_PER_DS = None   # None = todos com >= MIN_CONTEXTS válidos

print('Parâmetros carregados.')

Parâmetros carregados.


## 2. Ground truth de anotações (Siena e Mendeley)

In [3]:
# ── Horas de crise do Siena (relógio HH:MM:SS) ───────────────────────────────
# onset_s é calculado automaticamente pela função siena_seizures_from_clock()
# que lê o rec_start do header EDF e faz: onset = (sz_clock - rec_start) % 86400
# Nota: PN00-3.edf excluído (delta 61min = gravação inteira, não crise)
#       PN03-1.edf excluído (arquivo truncado — crise fora do range)
#       PN07/PN11: 1 crise apenas → nunca elegíveis (< MIN_CONTEXTS)
SIENA_SEIZURE_CLOCK = {
    'PN00': {
        'pn00-1.edf': [('19:58:36','19:59:46')],
        'pn00-2.edf': [('02:38:37','02:39:31')],
        # pn00-3.edf: IGNORADO — delta=61min, provavelmente intervalo de gravação
        'pn00-4.edf': [('21:08:29','21:09:43')],
        'pn00-5.edf': [('22:37:08','22:38:15')],
    },
    'PN01': {  # já em SIENA_GT; valores calculados = verificados ✓
        'pn01-1.edf': [('21:51:02','21:51:56'), ('07:53:17','07:54:31')],
    },
    'PN03': {
        # pn03-1.edf: EXCLUIDO (arquivo truncado)
        'pn03-2.edf': [('07:13:05','07:15:18')],
    },
    'PN05': {
        'pn05-2.edf': [('08:45:25','08:46:00')],
        'pn05-3.edf': [('07:55:19','07:55:49')],
        'pn05-4.edf': [('07:38:43','07:39:22')],
    },
    'PN06': {
        'pno6-1.edf': [('05:54:25','05:55:29')],  # atenção: PNO6 (O maiúsculo)
        'pno6-2.edf': [('23:39:09','23:40:18')],
        'pn06-3.edf': [('08:10:26','08:11:08')],
        'pno6-4.edf': [('12:55:08','12:56:11')],
        'pn06-5.edf': [('14:44:24','14:45:08')],
    },
    'PN09': {
        'pn09-1.edf': [('16:09:43','16:11:03')],
        'pn09-2.edf': [('17:00:56','17:01:55')],
        'pn09-3.edf': [('16:20:44','16:21:48')],
    },
    'PN10': {
        'pn10-1.edf':     [('07:45:50','07:46:59')],
        'pn10-2.edf':     [('11:40:13','11:41:04')],
        'pn10-3.edf':     [('15:43:53','15:45:02')],
        'pn10-4.5.6.edf': [('12:49:50','12:49:55'),('14:00:25','14:00:44'),('15:18:26','15:19:23')],
        'pn10-7.8.9.edf': [('17:35:13','17:36:01'),('18:20:24','18:20:42'),('20:24:48','20:25:03')],
        'pn10-10.edf':    [('10:58:19','10:58:33')],
    },
    'PN12': {
        'pn12-1.2.edf': [('16:13:23','16:14:26'), ('18:31:01','18:32:09')],
        'pn12-3.edf':   [('08:55:27','08:57:03')],
        'pn12-4.edf':   [('18:42:51','18:43:54')],
    },
    'PN13': {
        'pn13-1.edf': [('10:22:10','10:22:58')],
        'pn13-2.edf': [('08:55:51','08:56:56')],
        'pn13-3.edf': [('14:05:54','14:08:25')],
    },
    'PN14': {
        'pn14-1.edf': [('13:46:00','13:46:27')],
        'pn14-2.edf': [('17:54:52','17:55:04')],
        'pn14-3.edf': [('21:10:05','21:10:46')],
        'pn14-4.edf': [('15:49:33','15:50:56')],
    },
    'PN16': {
        'pn16-1.edf': [('22:45:05','22:47:08')],
        'pn16-2.edf': [('03:16:49','03:18:36')],
    },
    'PN17': {
        'pn17-1.edf': [('22:34:48','22:35:58')],
        'pn17-2.edf': [('16:01:09','16:02:32')],
    },
}

def _hms_to_s(hms_str):
    '''HH:MM:SS ou HH.MM.SS → segundos desde meia-noite.'''
    parts = hms_str.replace('.', ':').split(':')
    return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])

def parse_siena_annotation_txt(pat_dir):
    '''Lê Seizures-list-PNXX.txt do Siena sem abrir EDFs.

    Formato PhysioNet (verificado em PN10):
      Seizure n 1:
      File name: PN10-1.edf
      Registration start time: 05.40.05
      Seizure start time: 07.45.50
      Seizure end time: 07.46.59

    Quirks tratados:
      - "1 6.49.25" (typo de espaço) → "16.49.25"
      - "(CLINICAL ONSET)", "opure HH.MM.SS" (texto extra) → ignorado
      - Múltiplas crises no mesmo EDF → lista de (onset_s, offset_s)

    Retorna {edf_name_lower: [(onset_s, offset_s), ...]}
    '''
    pat_dir = Path(pat_dir)
    ann = None
    for pat_glob in ['Seizures-list-*.txt', 'Seizure*list*.txt', '*.txt']:
        txts = sorted(pat_dir.glob(pat_glob))
        if txts: ann = txts[0]; break
    if ann is None:
        return {}

    def _parse_time(t_str):
        t_str = t_str.strip()
        # Remove texto extra: "(CLINICAL ONSET)", "opure HH.MM.SS", etc.
        t_str = re.split(r'[;(]|\bopure\b|\bor\b', t_str, flags=re.I)[0].strip()
        # Fix typo: "1 6.49.25" → "16.49.25"
        t_str = re.sub(r'(\d)\s+(\d)', r'\1\2', t_str)
        # Normaliza separadores (. ou - → :)
        t_str = re.sub(r'[.\-]', ':', t_str)
        m = re.match(r'(\d{1,2}):(\d{2}):(\d{2})', t_str)
        if m:
            return int(m.group(1))*3600 + int(m.group(2))*60 + int(m.group(3))
        return None

    text   = ann.read_text(errors='ignore')
    result = {}  # {edf_lower: [(onset_s, offset_s)]}

    # Divide por bloco de crise ("Seizure n X")
    blocks = re.split(r'Seizure\s+n[°.]?\s*\d+', text, flags=re.IGNORECASE)
    for block in blocks[1:]:
        fname = rec_s = sz_on = sz_off = None
        for ln in block.split('\n'):
            ln = ln.strip()
            m = re.search(r'(?i)file\s+name\s*:\s*(\S+\.edf)', ln)
            if m: fname = m.group(1).lower(); continue
            m = re.search(r'(?i)registration\s+start\s+time\s*:\s*(.+)', ln)
            if m: rec_s = _parse_time(m.group(1)); continue
            m = re.search(r'(?i)seizure\s+start\s+time\s*:\s*(.+)', ln)
            if m: sz_on = _parse_time(m.group(1)); continue
            m = re.search(r'(?i)seizure\s+end\s+time\s*:\s*(.+)', ln)
            if m: sz_off = _parse_time(m.group(1)); continue
        if fname and rec_s is not None and sz_on is not None and sz_off is not None:
            onset_s  = (sz_on  - rec_s + 86400) % 86400
            offset_s = (sz_off - rec_s + 86400) % 86400
            if offset_s < onset_s: offset_s += 86400
            if 0 < offset_s - onset_s < 600:   # crise válida: 0–10min
                result.setdefault(fname, []).append((int(onset_s), int(offset_s)))
    return result


print('parse_siena_annotation_txt() definida (lê anotação sem abrir EDF).')
# Siena: onsets em segundos relativos ao inicio do arquivo
# Calculados como: onset = hms(sz_start) - hms(reg_start); se negativo: += 86400
# PN03-1.edf EXCLUIDO: arquivo truncado (349min em disco, crise em 644min)
SIENA_GT = {
    'PN01': {
        'pn01-1.edf': [(10218, 10272), (46353, 46427)],
    },
    'PN05': {
        'pn05-2.edf': [(7163, 7198)],
        'pn05-3.edf': [(6836, 6866)],
        'pn05-4.edf': [(3608, 3647)],
    },
    'PN09': {
        'pn09-1.edf': [(7249, 7329)],
        'pn09-2.edf': [(7127, 7186)],
        'pn09-3.edf': [(7221, 7285)],
    },
    'PN10': {
        'pn10-1.edf':       [(7545, 7614)],
        'pn10-2.edf':       [(7798, 7849)],
        'pn10-3.edf':       [(7835, 7904)],
        'pn10-4.5.6.edf':   [(2309, 2314), (6544, 6563), (11225, 11282)],
        'pn10-7.8.9.edf':   [(2748, 2796), (5459, 5477), (12923, 12938)],
        'pn10-10.edf':      [(7977, 7991)],
    },
    'PN13': {
        'pn13-1.edf': [(7062, 7110)],
        'pn13-2.edf': [(7249, 7314)],
        'pn13-3.edf': [(7553, 7704)],
    },
    'PN14': {
        'pn14-1.edf': [(7262,  7289)],
        'pn14-2.edf': [(7479,  7491)],
        'pn14-3.edf': [(17540, 17581)],
        'pn14-4.edf': [(5463,  5546)],
    },
    'PN17': {
        'pn17-1.edf': [(8420, 8490)],
        'pn17-2.edf': [(7731, 7814)],
    },
}

MENDELEY_GT = {
    'p10': {
        'p10_record1.edf': [(7199, 7644)],
        'p10_record2.edf': [(7200, 7505)],
    },
    'p13': {
        'p13_record1.edf': [(7196, 7248)],
        'p13_record2.edf': [(3241, 3266), (7196, 7212)],
        'p13_record3.edf': [(7240, 7258)],
        'p13_record4.edf': [(1450, 1480), (6651, 6675)],
    },
    'p15': {
        'p15_record1.edf': [(7158, 7208)],
        'p15_record2.edf': [(7147, 7194)],
        'p15_record3.edf': [(7132, 7145)],
        'p15_record4.edf': [(2431, 2487), (7234, 7254)],
    },
}
print('Ground truth carregado.')

parse_siena_annotation_txt() definida (lê anotação sem abrir EDF).
Ground truth carregado.


## 3. Funções auxiliares

In [4]:
def m(s):
    return round(s / 60, 1) if s is not None else None

def fmt_hms(s):
    if s is None: return '--'
    s = int(round(float(s)))
    return f'{s//3600:02d}:{(s%3600)//60:02d}:{s%60:02d}'

def normalize_fn(f):
    return re.sub(r'-+', '-', f.lower())

def siena_match(fn, gt):
    k = normalize_fn(fn)
    if k in gt: return gt[k]
    parts = k.split('-', 1)
    if len(parts) == 2:
        p, s = parts
        for v in [p.replace('0','o'), p.replace('o','0')]:
            if v+'-'+s in gt: return gt[v+'-'+s]
    return []

def parse_chbmit_summary(path):
    sz_map, free = {}, []
    blocks = open(path, errors='ignore').read().split('File Name:')
    for block in blocks[1:]:
        lines = block.strip().split('\n')
        fname = lines[0].strip()
        n_sz = 0; onsets = []; offsets = []
        for ln in lines:
            if 'Number of Seizures' in ln:
                try: n_sz = int(re.search(r'\d+', ln.split(':',1)[1]).group())
                except: pass
            elif 'Seizure' in ln and 'Start Time' in ln:
                mx = re.search(r'(\d+)', ln.split(':',1)[1])
                if mx: onsets.append(int(mx.group()))
            elif 'Seizure' in ln and 'End Time' in ln:
                mx = re.search(r'(\d+)', ln.split(':',1)[1])
                if mx: offsets.append(int(mx.group()))
        if n_sz > 0:
            sz_map[fname] = [(o,e) for o,e in zip(onsets,offsets) if e>o]
        else:
            free.append(fname)
    return sz_map, free

def get_edf_duration(path):
    if HAS_MNE:
        try:
            raw = mne.io.read_raw_edf(str(path), preload=False, verbose=False)
            d = (raw.n_times - 1) / raw.info['sfreq']
            raw.close()
            return d
        except: pass
    return path.stat().st_size / (256 * 23 * 2)

print('Funções auxiliares definidas.')

Funções auxiliares definidas.


## 4. Validadores de contexto

In [5]:
def validate_dedicated(pat, edf_list):
    '''CHB-MIT / SeizeIT2: interictal de arquivos puros, ponteiro sequencial.
    Criterio: onset >= MAX_PRE_S (40min).
    Pre-ictal verificado contra pos-ictal de crises anteriores no mesmo arquivo.
    '''
    pure_edfs = sorted([e for e in edf_list if e['is_pure']], key=lambda x: x['name'])
    sz_edfs   = sorted([e for e in edf_list if not e['is_pure'] and e['seizures']],
                       key=lambda x: x['name'])
    pool = [{'name': pe['name'], 'path': pe['path'],
             'dur_s': pe['dur_s'], 'used_s': 0.0} for pe in pure_edfs]
    pool_idx = 0
    contexts = []

    for edf in sz_edfs:
        seiz = edf['seizures']
        for i, (on, off) in enumerate(seiz):
            approved = on >= MAX_PRE_S
            motivo = None
            if not approved:
                motivo = f'onset={m(on)}min < MAX_PRE={m(MAX_PRE_S)}min'
            else:
                pre_ini = on - MAX_PRE_S
                for j, (on_o, off_o) in enumerate(seiz):
                    if j == i: continue
                    if pre_ini < off_o and on > on_o:
                        approved = False
                        motivo = f'pre-ictal contem crise #{j+1}'
                        break
                    if off_o < on and pre_ini < off_o + GUARD_S and on > off_o:
                        approved = False
                        motivo = f'pre-ictal colide pos-ictal crise #{j+1}'
                        break

            # calcula pre_max_disp
            prev_limit = max((off_o + GUARD_S for _, off_o in seiz if off_o < on),
                             default=0)
            pre_max_disp = max(0, on - prev_limit)

            if not approved:
                contexts.append({
                    'edf': edf['name'], 'path': edf['path'],
                    'dur_edf': edf['dur_s'], 'sz_idx': i+1,
                    'on_s': on, 'off_s': off, 'n_sz_edf': len(seiz),
                    'approved_pre': False, 'pre_max_disp_s': pre_max_disp,
                    'pre_inicio_s': None, 'pre_fim_s': None,
                    'inter_arquivo': None, 'inter_path': None,
                    'inter_inicio_s': None, 'inter_fim_s': None,
                    'inter_fonte': '--', 'valid': False, 'motivo': motivo,
                })
                continue

            pi = pool_idx
            found = False
            while pi < len(pool):
                p = pool[pi]
                restante = p['dur_s'] - p['used_s']
                if restante >= MIN_INTER_S:
                    # CHB-MIT: salva arquivo puro inteiro (tipicamente 60min).
                    # 1 arquivo por contexto — sem compartilhamento para evitar leakage.
                    ini = p['used_s']; fim = p['dur_s']
                    p['used_s'] = fim; pool_idx = pi
                    contexts.append({
                        'edf': edf['name'], 'path': edf['path'],
                        'dur_edf': edf['dur_s'], 'sz_idx': i+1,
                        'on_s': on, 'off_s': off, 'n_sz_edf': len(seiz),
                        'approved_pre': True, 'pre_max_disp_s': pre_max_disp,
                        'pre_inicio_s': on - MAX_PRE_S, 'pre_fim_s': on,
                        'inter_arquivo': p['name'], 'inter_path': p['path'],
                        'inter_inicio_s': ini, 'inter_fim_s': fim,
                        'inter_fonte': 'dedicado',
                        'valid': True,
                        'motivo': f'inter dedicado {p["name"]} [{m(ini)}-{m(fim)}min]',
                    })
                    found = True; break
                pi += 1; pool_idx = pi
            if not found:
                contexts.append({
                    'edf': edf['name'], 'path': edf['path'],
                    'dur_edf': edf['dur_s'], 'sz_idx': i+1,
                    'on_s': on, 'off_s': off, 'n_sz_edf': len(seiz),
                    'approved_pre': True, 'pre_max_disp_s': pre_max_disp,
                    'pre_inicio_s': on - MAX_PRE_S, 'pre_fim_s': on,
                    'inter_arquivo': None, 'inter_path': None,
                    'inter_inicio_s': None, 'inter_fim_s': None,
                    'inter_fonte': 'esgotado', 'valid': False,
                    'motivo': 'pool de interictal puro esgotado',
                })
    return contexts

In [6]:
def validate_intra(pat, edf_list):
    '''Siena / Mendeley: interictal intra-arquivo, 2 passadas + lock.
    Criterio: onset >= MAX_PRE_S + GUARD_S (70min).
    '''
    MIN_ONSET = MAX_PRE_S + GUARD_S
    contexts = []

    for edf in edf_list:
        if edf['is_pure'] or not edf['seizures']:
            continue
        dur  = edf['dur_s']
        seiz = edf['seizures']

        # Passada 1: zonas proibidas de todas as crises
        forbidden = [(max(0, on - MAX_PRE_S - GUARD_S), off + GUARD_S)
                     for on, off in seiz]
        forbidden.sort()
        locked = []

        def free_space(start, end):
            blocks = []; cur = start
            for (a, b) in sorted(forbidden + locked):
                if b <= cur: continue
                if a >= end: break
                if a > cur: blocks.append((cur, a))
                cur = max(cur, b)
            if cur < end: blocks.append((cur, end))
            return blocks

        # Passada 2: aloca por crise
        for i, (on, off) in enumerate(seiz):
            zona_ini = max(0, on - MAX_PRE_S - GUARD_S)
            zona_fim = off + GUARD_S  # dur pode ser None

            prev_limit = max((off_o + GUARD_S for _, off_o in seiz if off_o < on),
                             default=0)
            pre_max_disp = max(0, on - prev_limit)

            if on < MIN_ONSET:
                contexts.append({
                    'edf': edf['name'], 'path': edf['path'],
                    'dur_edf': dur, 'sz_idx': i+1,
                    'on_s': on, 'off_s': off, 'n_sz_edf': len(seiz),
                    'approved_pre': False, 'pre_max_disp_s': pre_max_disp,
                    'pre_inicio_s': None, 'pre_fim_s': None,
                    'inter_arquivo': None, 'inter_path': None,
                    'inter_inicio_s': None, 'inter_fim_s': None,
                    'inter_fonte': '--', 'valid': False,
                    'motivo': f'onset={m(on)}min < {m(MIN_ONSET)}min (MAX_PRE+GUARD)',
                }); continue

            pre_ini = on - MAX_PRE_S
            pre_colide = any(off_o < on and pre_ini < off_o + GUARD_S and on > off_o
                             for _, off_o in seiz)
            if pre_colide:
                contexts.append({
                    'edf': edf['name'], 'path': edf['path'],
                    'dur_edf': dur, 'sz_idx': i+1,
                    'on_s': on, 'off_s': off, 'n_sz_edf': len(seiz),
                    'approved_pre': False, 'pre_max_disp_s': pre_max_disp,
                    'pre_inicio_s': None, 'pre_fim_s': None,
                    'inter_arquivo': None, 'inter_path': None,
                    'inter_inicio_s': None, 'inter_fim_s': None,
                    'inter_fonte': '--', 'valid': False,
                    'motivo': 'pre-ictal colide com pos-ictal de crise anterior',
                }); continue

            r1 = free_space(0, zona_ini);     r1t = sum(b-a for a,b in r1)
            # r3 removido (pós-crise pertence ao próximo contexto)

            inter_ini = inter_fim = inter_path = None; fonte = None
            # Seleciona o ÚLTIMO segmento contíguo de r1 (= t_livre_efetivo → guard_ini).
            # r1 pode ter múltiplos segmentos se houver crises intermediárias no arquivo
            # (comum no SeizeIT2 com muitas crises por run).
            # Usar bounding box r1[0]→r1[-1] incluiria essas crises intermediárias.
            # O último segmento é: (pós-ictal_da_última_crise_intermediária, zona_ini)
            # = exatamente t_livre_efetivo até guard_ini, garantidamente livre de crises.
            last_seg = r1[-1] if r1 else None
            last_seg_dur = (last_seg[1] - last_seg[0]) if last_seg else 0
            if last_seg_dur >= MIN_INTER_S:
                inter_ini = last_seg[0]   # t_livre_efetivo
                inter_fim = last_seg[1]   # onset - MAX_PRE_S - GUARD_S
                inter_path = edf['path']; fonte = 'antes (R1-último-segmento)'

            if fonte:
                locked.append((inter_ini, inter_fim)); locked.sort()
                contexts.append({
                    'edf': edf['name'], 'path': edf['path'],
                    'dur_edf': dur, 'sz_idx': i+1,
                    'on_s': on, 'off_s': off, 'n_sz_edf': len(seiz),
                    'approved_pre': True, 'pre_max_disp_s': pre_max_disp,
                    'pre_inicio_s': on - MAX_PRE_S, 'pre_fim_s': on,
                    'inter_arquivo': edf['name'], 'inter_path': inter_path,
                    'inter_inicio_s': inter_ini, 'inter_fim_s': inter_fim,
                    'inter_fonte': fonte, 'valid': True,
                    'motivo': f'{fonte}: inter=[{m(inter_ini)}-{m(inter_fim)}min] pre=[{m(on-MAX_PRE_S)}-{m(on)}min]',
                })
            else:
                contexts.append({
                    'edf': edf['name'], 'path': edf['path'],
                    'dur_edf': dur, 'sz_idx': i+1,
                    'on_s': on, 'off_s': off, 'n_sz_edf': len(seiz),
                    'approved_pre': True, 'pre_max_disp_s': pre_max_disp,
                    'pre_inicio_s': on - MAX_PRE_S, 'pre_fim_s': on,
                    'inter_arquivo': None, 'inter_path': None,
                    'inter_inicio_s': None, 'inter_fim_s': None,
                    'inter_fonte': 'insuf', 'valid': False,
                    'motivo': f'inter insuf: R1={m(r1t)}min R3={m(r3t)}min',
                })
    return contexts

print('Validadores definidos.')

Validadores definidos.


## 5. Carregamento dos EDFs por paciente

In [7]:

def discover_all_patients(verbose=True):
    """Descobre automaticamente todos os pacientes disponíveis em disco.

    Para cada dataset, percorre o diretório raiz e identifica os pacientes
    com base na estrutura de arquivos esperada (summary.txt para CHB-MIT,
    diretório de EDF para Siena/SeizeIT2/Mendeley).
    """
    found = {'CHBMIT': [], 'Siena': [], 'SeizeIT2': [], 'Mendeley': []}

    # CHB-MIT: subpastas chbXX com summary.txt
    if CHBMIT_DIR.exists():
        pats = sorted(
            d.name for d in CHBMIT_DIR.iterdir()
            if d.is_dir() and d.name.startswith('chb')
            and (d / f'{d.name}-summary.txt').exists()
        )
        found['CHBMIT'] = pats
        if verbose: print(f'CHB-MIT:  {len(pats)} pacientes em disco')

    # Siena: subpastas PNXX com pelo menos 1 EDF
    if SIENA_DIR.exists():
        pats = sorted(
            d.name for d in SIENA_DIR.iterdir()
            if d.is_dir() and d.name.startswith('PN')
            and any(d.glob('*.edf'))
        )
        found['Siena'] = pats
        if verbose: print(f'Siena:    {len(pats)} pacientes em disco')

    # SeizeIT2: subpastas sub-XXX com pasta ses-01/eeg/
    if SEIZEIT_DIR.exists():
        pats = sorted(
            d.name for d in SEIZEIT_DIR.iterdir()
            if d.is_dir() and d.name.startswith('sub-')
            and (d / SEIZEIT_SESSION / 'eeg').exists()
        )
        found['SeizeIT2'] = pats
        if verbose: print(f'SeizeIT2: {len(pats)} pacientes em disco')

    # Mendeley: pacientes identificados pelos prefixos nos nomes dos EDFs
    if MENDELEY_DIR.exists():
        edf_dir = MENDELEY_DIR / 'Raw_EDF_Files'
        if not edf_dir.exists(): edf_dir = MENDELEY_DIR
        prefixes = set()
        for f in edf_dir.glob('*.edf'):
            # nomes como p10_record1.edf → prefixo = p10
            m = re.match(r'^(p\d+)_', f.name.lower())
            if m: prefixes.add(m.group(1))
        pats = sorted(prefixes)
        found['Mendeley'] = pats
        if verbose: print(f'Mendeley: {len(pats)} pacientes em disco')

    total = sum(len(v) for v in found.values())
    if verbose: print(f'Total em disco: {total} pacientes')
    return found

def load_patient_edfs(dataset, pat):
    edf_list = []
    if dataset == 'CHBMIT':
        pat_dir = CHBMIT_DIR / pat
        summary = pat_dir / f'{pat}-summary.txt'
        if not summary.exists(): return []
        sz_map, free_list = parse_chbmit_summary(summary)
        disk = {f.name: f for f in pat_dir.glob('*.edf')}
        for fn, iv in sz_map.items():
            if fn in disk:
                edf_list.append({'name':fn,'path':disk[fn],
                                 'dur_s':get_edf_duration(disk[fn]),
                                 'seizures':iv,'is_pure':False})
        for fn in free_list:
            if fn in disk:
                edf_list.append({'name':fn,'path':disk[fn],
                                 'dur_s':get_edf_duration(disk[fn]),
                                 'seizures':[],'is_pure':True})

    elif dataset == 'Siena':
        pat_dir = SIENA_DIR / pat
        # 1. SIENA_GT: onset_s verificados manualmente (prioridade máxima)
        gt_s = SIENA_GT.get(pat, {})
        # 2. Fallback: parse_siena_annotation_txt — lê rec_start do txt de anotação
        #    sem abrir nenhum EDF. Suporta formato PhysioNet com "Registration start time".
        ann_map = parse_siena_annotation_txt(pat_dir)  # {edf_name_lower: [(on,off)]}
        for ep in sorted(pat_dir.glob('*.edf')):
            fn_corr = fix_siena_filename(ep.name)
            iv = siena_match(fn_corr, gt_s)       # GT verificado
            if not iv:
                iv = ann_map.get(ep.name.lower(), [])  # anotação txt
            edf_list.append({'name': ep.name, 'path': ep,
                             'dur_s': None,  # intra: dur_s não necessário
                             'seizures': iv, 'is_pure': len(iv) == 0})
    elif dataset == 'SeizeIT2':
        eeg_dir = SEIZEIT_DIR / pat / SEIZEIT_SESSION / 'eeg'
        if not eeg_dir.exists(): return []
        for ep in sorted(eeg_dir.glob('*_eeg.edf')):
            tsv = eeg_dir / ep.name.replace('_eeg.edf','_events.tsv')
            iv = []
            if tsv.exists():
                try:
                    df = pd.read_csv(tsv, sep='\t')
                    if 'eventType' in df.columns:
                        et = df['eventType'].astype(str).str.lower().str.strip()
                        for _, row in df[et.str.startswith('sz')].iterrows():
                            try:
                                o = float(row['onset']); d = float(row['duration'])
                                if d > 0: iv.append((o, o+d))
                            except: pass
                except: pass
            edf_list.append({'name':ep.name,'path':ep,
                             'dur_s':get_edf_duration(ep),
                             'seizures':iv,'is_pure':len(iv)==0})

    elif dataset == 'Mendeley':
        edf_dir = MENDELEY_DIR / 'Raw_EDF_Files'
        if not edf_dir.exists(): edf_dir = MENDELEY_DIR
        gt = MENDELEY_GT.get(pat, {})
        for ep in sorted(edf_dir.glob(f'{pat}_*.edf')):
            iv = gt.get(ep.name.lower(), [])
            edf_list.append({'name':ep.name,'path':ep,
                             'dur_s':None,  # intra: dur não necessário após remoção do r3
                             'seizures':iv,'is_pure':len(iv)==0})
    return edf_list

print('Loader definido.')

Loader definido.


## 6. Validação de todos os pacientes selecionados

In [8]:
# Estratégia de interictal por dataset:
# INTRA-ARQUIVO: inter = EEG antes da crise no mesmo arquivo (t_livre → guard_ini)
#   → Siena, Mendeley: gravações de sessão hospitalar, crise no final do arquivo
#   → SeizeIT2: monitorização domiciliar contínua (horas), crise em algum ponto
# DEDICADO: inter = arquivo puro de sessão separada (outro dia)
#   → CHB-MIT: arquivos de 60min gravados em sessões sem crise
VALIDATORS = {
    'CHBMIT':   validate_dedicated,   # sessões separadas → pool de arquivos puros
    'SeizeIT2': validate_intra,        # gravações contínuas domiciliares → r1 completo
    'Siena':    validate_intra,        # sessão hospitalar → r1 completo
    'Mendeley': validate_intra,        # sessão hospitalar → r1 completo
}

# ── Passo 1: descoberta automática ───────────────────────────────────────────
print('Descobrindo pacientes em disco...')
ALL_IN_DISK = discover_all_patients(verbose=True)

# ── Passo 2: valida TODOS os pacientes disponíveis ────────────────────────────
all_contexts_full = {}   # todos os pacientes (elegíveis e não-elegíveis)
all_edfs          = {}

for dataset, pats in ALL_IN_DISK.items():
    all_contexts_full[dataset] = {}
    all_edfs[dataset]          = {}
    print(f'\n=== {dataset} ({len(pats)} pacientes) ===')
    for pat in pats:
        edf_list = load_patient_edfs(dataset, pat)
        all_edfs[dataset][pat] = edf_list
        if not edf_list:
            print(f'  {pat}: sem EDFs em disco'); continue
        ctxs = VALIDATORS[dataset](pat, edf_list)
        all_contexts_full[dataset][pat] = ctxs
        n_valid = sum(1 for c in ctxs if c['valid'])
        n_total = len(ctxs)
        tag = 'OK' if n_valid >= MIN_CONTEXTS else 'INSUF'
        print(f'  [{tag}] {pat}: {n_total} crises | {n_valid} ctx validos')

# ── Passo 3: filtra (>= MIN_CONTEXTS) e rankeia por n_valid desc ──────────────
print('\n' + '='*60)
print('SELEÇÃO AUTOMÁTICA — ranking por contextos válidos')
print('='*60)

all_contexts = {}   # apenas os selecionados
SELECTED     = {}   # dicionário final (usado pelo resto do notebook)

for dataset, pat_ctxs in all_contexts_full.items():
    # Elegíveis = >= MIN_CONTEXTS válidos
    eligible = [
        (pat, ctxs, sum(1 for c in ctxs if c['valid']))
        for pat, ctxs in pat_ctxs.items()
        if sum(1 for c in ctxs if c['valid']) >= MIN_CONTEXTS
    ]
    # Ordena por n_valid descrescente (mais contextos primeiro)
    eligible.sort(key=lambda x: -x[2])

    # Aplica cap se N_SELECT_PER_DS estiver definido
    if N_SELECT_PER_DS is not None:
        eligible = eligible[:N_SELECT_PER_DS]

    all_contexts[dataset] = {pat: ctxs for pat, ctxs, _ in eligible}
    SELECTED[dataset]     = [pat for pat, _, _ in eligible]

    print(f'\n{dataset}: {len(eligible)} selecionados de '
          f'{len(pat_ctxs)} validados')
    for pat, ctxs, n_v in eligible:
        n_t = len(ctxs)
        print(f'  #{eligible.index((pat,ctxs,n_v))+1:02d}  {pat:<14} '
              f'{n_v:3d} ctx válidos / {n_t:3d} crises')

total_pats = sum(len(v) for v in SELECTED.values())
total_ctx  = sum(sum(1 for c in ctxs if c['valid'])
                 for ds in all_contexts.values()
                 for ctxs in ds.values())
print(f'\nTotal selecionados: {total_pats} pacientes | {total_ctx} contextos')

Descobrindo pacientes em disco...
Total em disco: 0 pacientes

=== CHBMIT (0 pacientes) ===

=== Siena (0 pacientes) ===

=== SeizeIT2 (0 pacientes) ===

=== Mendeley (0 pacientes) ===

SELEÇÃO AUTOMÁTICA — ranking por contextos válidos

CHBMIT: 0 selecionados de 0 validados

Siena: 0 selecionados de 0 validados

SeizeIT2: 0 selecionados de 0 validados

Mendeley: 0 selecionados de 0 validados

Total selecionados: 0 pacientes | 0 contextos


## 7. Verificação de integridade

In [9]:
def check_integrity(dataset, pat, edf_list, contexts):
    problems = []
    sz_by_file = {e['name']: e['seizures'] for e in edf_list}
    valid_ctxs = [c for c in contexts if c['valid']]

    for c in valid_ctxs:
        if c['inter_inicio_s'] is None: continue
        arq = c['inter_arquivo']
        ii, if_ = c['inter_inicio_s'], c['inter_fim_s']
        for (con, coff) in sz_by_file.get(arq, []):
            if ii < coff and if_ > con:
                problems.append(f'{pat} ctx#{c["sz_idx"]}: interictal contem crise em {arq}')

    for c in valid_ctxs:
        if c['pre_inicio_s'] is None: continue
        arq = c['edf']
        pre_i, pre_f = c['pre_inicio_s'], c['pre_fim_s']
        for (con, coff) in sz_by_file.get(arq, []):
            if coff >= c['on_s']: continue
            if pre_i < coff + GUARD_S and pre_f > coff:
                problems.append(f'{pat} ctx#{c["sz_idx"]}: pre-ictal colide pos-ictal anterior')

    inter_by_file = defaultdict(list)
    for c in valid_ctxs:
        if c['inter_inicio_s'] is not None:
            inter_by_file[c['inter_arquivo']].append(
                (c['inter_inicio_s'], c['inter_fim_s'], c['sz_idx']))
    for arq, ivs in inter_by_file.items():
        ivs.sort()
        for a in range(len(ivs)-1):
            if ivs[a][1] > ivs[a+1][0]:
                problems.append(f'{pat}: interictais sobrepostos em {arq}: '
                                 f'ctx#{ivs[a][2]} e ctx#{ivs[a+1][2]}')
    return problems

print('VERIFICAÇÃO DE INTEGRIDADE')
print('='*55)
total_problems = 0
for dataset, pats in SELECTED.items():
    for pat in pats:
        if pat not in all_contexts.get(dataset, {}): continue
        probs = check_integrity(dataset, pat,
                                all_edfs[dataset][pat],
                                all_contexts[dataset][pat])
        if probs:
            total_problems += len(probs)
            for p in probs: print(f'  PROBLEMA: {p}')

if total_problems == 0:
    print('TUDO OK — nenhuma violação encontrada.')
else:
    print(f'{total_problems} problema(s) encontrado(s).')

VERIFICAÇÃO DE INTEGRIDADE
TUDO OK — nenhuma violação encontrada.


## 8. Tabela detalhada de contextos por paciente

Para cada paciente: todos os contextos (válidos e inválidos), com:
- Arquivo EDF da crise e duração
- Onset e offset (min)
- Tipo de interictal (dedicado / R1 / R3 / R1+R3)
- Arquivo fonte do interictal e trecho exato [início – fim] em min
- Pré-ictal máximo disponível
- Status do contexto e motivo

In [10]:
def print_tabela_paciente(dataset, pat, contexts):
    n_valid = sum(1 for c in contexts if c['valid'])
    tipo_ds = 'dedicado' if dataset in ('CHBMIT',) else 'intra-arquivo'
    print(f'\n' + '─'*80)
    print(f'  {dataset} / {pat}  [{tipo_ds}]  —  {n_valid}/{len(contexts)} contextos válidos')
    print('─'*80)
    ctx_num = 0
    for c in contexts:
        if c['valid']: ctx_num += 1
        status = f'✓ CONTEXTO {ctx_num}' if c['valid'] else '✗ inválido'
        print(f'  Crise #{c["sz_idx"]:2d} | {c["edf"]}  ({m(c["dur_edf"]):.0f}min)')
        print(f'    Onset : {m(c["on_s"]):7.1f} min  ({fmt_hms(c["on_s"])})')
        print(f'    Offset: {m(c["off_s"]):7.1f} min  ({fmt_hms(c["off_s"])})  '
              f'dur={m(c["off_s"]-c["on_s"]):.1f}min')
        print(f'    Pré-ictal máx disp: {m(c["pre_max_disp_s"]):.1f} min', end='')
        if c['approved_pre'] and c['pre_inicio_s'] is not None:
            print(f'  →  reservado [{m(c["pre_inicio_s"]):.1f} – {m(c["pre_fim_s"]):.1f}min]')
        else:
            print()
        if c['valid']:
            ii, if_ = c['inter_inicio_s'], c['inter_fim_s']
            print(f'    Interictal : {c["inter_fonte"]}  |  arquivo: {c["inter_arquivo"]}')
            print(f'                 [{m(ii):.1f} – {m(if_):.1f} min]  ({m(if_-ii):.0f}min)')
        else:
            print(f'    Motivo     : {c["motivo"]}')
        print(f'    Status     : {status}')

for dataset, pats in SELECTED.items():
    for pat in pats:
        if pat not in all_contexts.get(dataset, {}): continue
        print_tabela_paciente(dataset, pat, all_contexts[dataset][pat])

# Resumo consolidado
print('\n' + '='*80)
print('RESUMO CONSOLIDADO')
print('='*80)
print(f'  {"Dataset":<12} {"Paciente":<12} {"Crises":<8} {"Ctx val":<10} '
      f'{"Selecionável?":<15} {"Fonte inter"}')
print('  ' + '-'*74)
total_sel = 0; total_ctx_val = 0; total_crises = 0
for dataset, pats in SELECTED.items():
    for pat in pats:
        if pat not in all_contexts.get(dataset, {}): continue
        ctxs = all_contexts[dataset][pat]
        n_val = sum(1 for c in ctxs if c['valid'])
        n_tot = len(ctxs)
        sel = 'Sim ✓' if n_val >= MIN_CONTEXTS else 'Não ✗'
        tipo = 'dedicado' if dataset in ('CHBMIT',) else 'intra-arquivo'
        print(f'  {dataset:<12} {pat:<12} {n_tot:<8} {n_val:<10} {sel:<15} {tipo}')
        if n_val >= MIN_CONTEXTS: total_sel += 1
        total_ctx_val += n_val; total_crises += n_tot
print('  ' + '-'*74)
print(f'  {"TOTAL":<12} {"24 pac":<12} {total_crises:<8} {total_ctx_val:<10} '
      f'{str(total_sel)+" pac sel":<15}')
print(f'\nParâmetros: G={int(GUARD_S//60)}min  P={int(MAX_PRE_S//60)}min  I={int(MIN_INTER_S//60)}min  '
      f'MIN_CTX={MIN_CONTEXTS}')


RESUMO CONSOLIDADO
  Dataset      Paciente     Crises   Ctx val    Selecionável?   Fonte inter
  --------------------------------------------------------------------------
  --------------------------------------------------------------------------
  TOTAL        24 pac       0        0          0 pac sel      

Parâmetros: G=30min  P=40min  I=40min  MIN_CTX=2


## 9. Pré-processamento e salvamento dos sinais em .npz

Para cada contexto válido:
1. Carrega o segmento EDF (interictal e pré-ictal)
2. Aplica filtros: passa-alta 0.5Hz, passa-baixa 40Hz, notch 50/60Hz
3. Reamostra para TARGET_SFREQ (256Hz)
4. Salva em `data/signals/{dataset}__{pat}__ctx{N}.npz`

Cada `.npz` contém:
- `inter`: sinal interictal (canais × amostras) em µV
- `pre`: sinal pré-ictal (canais × amostras) em µV
- `ch_names`: nomes dos canais EEG selecionados
- `sfreq`: frequência de amostragem após reamostragem
- `meta`: dicionário com todos os metadados do contexto

In [11]:
# ── Parâmetros de pré-processamento ──────────────────────────────────────────
CLIP_UV       = 500.0    # Clipping robusto ±500µV (antes da filtragem)
                         # Protege o filtro IIR de ringing causado por artefactos
                         # de saturação do amplificador (movimento, eletrodo solto)
APPLY_CAR     = True     # Re-referência Common Average Reference (CAR)
                         # Aplicada apenas em montagens referenciais multi-canal
                         # (Siena, Mendeley). CHB-MIT (bipolar) e SeizeIT2 (2ch)
                         # excluídos — CAR em bipolar/2ch não tem sentido.
DATASETS_CAR  = {'Siena', 'Mendeley'}   # datasets que recebem CAR

# NOTA sobre Z-score (normalização por segmento):
# Não aplicada ao sinal bruto — removeria a amplitude absoluta (std, var, RMS)
# que é um biomarcador pré-ictal. A normalização de features é feita no NB4
# via StandardScaler (dentro do LOSO, sem leakage).

def load_and_preprocess(path, tmin_s, tmax_s, apply_car=False):
    '''Carrega trecho do EDF, aplica pré-processamento e retorna (data_uV, ch_names, sfreq).

    Pipeline:
      1. Clipping ±CLIP_UV antes da filtragem (protege filtro IIR de artefactos)
      2. Highpass / Lowpass / Notch
      3. Reamostragem para TARGET_SFREQ
      4. CAR (opcional, apenas montagens referenciais multi-canal)
    '''
    if not HAS_MNE or path is None:
        return None, None, None
    try:
        raw = mne.io.read_raw_edf(str(path), preload=False, verbose=False)
        sf  = raw.info['sfreq']

        # limita ao tamanho real
        tmin_s = max(0.0, float(tmin_s))
        tmax_s = min((raw.n_times - 1) / sf, float(tmax_s))
        if tmax_s <= tmin_s:
            raw.close(); return None, None, None

        # seleciona canais EEG
        non_eeg = {'spo2','hr','ekg','ecg','emg','eog','mk','status',
                   'annot','trig','stim','resp','temp'}
        eeg_idx = [i for i,ch in enumerate(raw.ch_names)
                   if ch.upper().startswith('EEG')]
        if not eeg_idx:
            eeg_idx = [i for i,ch in enumerate(raw.ch_names)
                       if not any(kw in ch.lower() for kw in non_eeg)]
        if not eeg_idx:
            eeg_idx = list(range(len(raw.ch_names)))

        ch_names = [raw.ch_names[i] for i in eeg_idx]

        # recorta e carrega em memoria
        raw_crop = raw.copy().crop(tmin=tmin_s, tmax=tmax_s)
        raw_crop.pick(eeg_idx)
        raw_crop.load_data(verbose=False)

        # ── PASSO 1: Clipping robusto ANTES da filtragem ──────────────────────
        # Artefactos de saturação (spikes de ±mV) distorcem o filtro IIR nas
        # janelas vizinhas via efeito de Gibbs. O clip em Volts equivale a CLIP_UV µV.
        clip_v = CLIP_UV * 1e-6   # converte µV → V (unidade do MNE)
        raw_crop._data = np.clip(raw_crop._data, -clip_v, clip_v)

        # ── PASSO 2: Filtragem ────────────────────────────────────────────────
        raw_crop.filter(l_freq=F_HIGHPASS, h_freq=F_LOWPASS,
                        method='iir', verbose=False)
        raw_crop.notch_filter(freqs=NOTCH_FREQS, verbose=False)

        # ── PASSO 3: Reamostragem ─────────────────────────────────────────────
        if abs(sf - TARGET_SFREQ) > 1:
            raw_crop.resample(TARGET_SFREQ, verbose=False)

        data = raw_crop.get_data() * 1e6  # V → µV
        sfreq_out = raw_crop.info['sfreq']
        raw_crop.close(); raw.close()

        # ── PASSO 4: CAR (Common Average Reference) ───────────────────────────
        # Subtrai a média espacial instantânea de todos os canais.
        # Remove artefactos de modo comum (ruído de alimentação, movimento do
        # cabo) e torna o sinal independente da referência do hardware.
        # Apenas para montagens referenciais com ≥8 canais (Siena, Mendeley).
        # CHB-MIT: montagem bipolar — CAR já está implícita; aplicar novamente
        #          introduziria artefactos.
        # SeizeIT2: apenas 2 canais — CAR daria (ch1-ch2)/2 e (ch2-ch1)/2,
        #           destruindo metade da informação útil.
        if apply_car and data.shape[0] >= 8:
            data -= data.mean(axis=0, keepdims=True)  # (n_ch, n_samp) → spatial mean

        return data, ch_names, sfreq_out
    except Exception as e:
        print(f'    Erro {Path(str(path)).name}: {e}')
        return None, None, None

def save_context_npz(dataset, pat, ctx_id, ctx):
    '''Salva o contexto em .npz pre-processado.'''
    out_path = SIGNAL_DIR / f'{dataset}__{pat}__ctx{ctx_id:03d}.npz'
    if out_path.exists():
        return True, 'ja existe'

    car = dataset in DATASETS_CAR
    inter_data, inter_chs, sfreq = load_and_preprocess(
        ctx['inter_path'], ctx['inter_inicio_s'], ctx['inter_fim_s'],
        apply_car=car)
    pre_data, pre_chs, _ = load_and_preprocess(
        ctx['path'], ctx['pre_inicio_s'], ctx['pre_fim_s'],
        apply_car=car)

    if inter_data is None or pre_data is None:
        return False, 'falha ao carregar EDF'

    # garante mesmos canais
    common = [c for c in inter_chs if c in pre_chs]
    if not common:
        common = inter_chs[:min(len(inter_chs), len(pre_chs))]
    inter_idx = [inter_chs.index(c) for c in common if c in inter_chs]
    pre_idx   = [pre_chs.index(c)   for c in common if c in pre_chs]
    if not inter_idx or not pre_idx:
        return False, 'sem canais em comum'

    np.savez_compressed(
        out_path,
        inter     = inter_data[inter_idx],
        pre       = pre_data[pre_idx],
        ch_names  = np.array(common),
        sfreq     = np.array(sfreq),
        meta      = np.array(json.dumps({
            'dataset': dataset, 'paciente': pat, 'contexto_id': ctx_id,
            'arquivo_crise': ctx['edf'], 'onset_s': ctx['on_s'],
            'offset_s': ctx['off_s'],
            'inter_arquivo': ctx['inter_arquivo'],
            'inter_inicio_s': ctx['inter_inicio_s'],
            'inter_fim_s': ctx['inter_fim_s'],
            'inter_fonte': ctx['inter_fonte'],
            'pre_inicio_s': ctx['pre_inicio_s'],
            'pre_fim_s': ctx['pre_fim_s'],
            'pre_max_disp_s': ctx['pre_max_disp_s'],
            'guard_s': GUARD_S, 'max_pre_s': MAX_PRE_S,
            'target_sfreq': TARGET_SFREQ,
        }))
    )
    return True, out_path.name

print('Funções de pré-processamento definidas.')

Funções de pré-processamento definidas.


In [12]:
# Salva todos os contextos válidos em .npz
print('Salvando contextos em .npz...\n')
saved = 0; failed = 0; skipped = 0
for dataset, pats in SELECTED.items():
    print(f'--- {dataset} ---')
    for pat in pats:
        if pat not in all_contexts.get(dataset, {}): continue
        ctx_id = 0
        for c in all_contexts[dataset][pat]:
            if not c['valid']: continue
            ctx_id += 1
            ok, msg = save_context_npz(dataset, pat, ctx_id, c)
            if 'ja existe' in msg: skipped += 1
            elif ok: saved += 1; print(f'  {pat} ctx{ctx_id:03d}: {msg}')
            else: failed += 1; print(f'  {pat} ctx{ctx_id:03d}: FALHOU — {msg}')

print(f'\nSalvos: {saved} | Já existiam: {skipped} | Falhas: {failed}')
print(f'Total .npz em {SIGNAL_DIR}: {len(list(SIGNAL_DIR.glob("*.npz")))}')

Salvando contextos em .npz...

--- CHBMIT ---
--- Siena ---
--- SeizeIT2 ---
--- Mendeley ---

Salvos: 0 | Já existiam: 0 | Falhas: 0
Total .npz em data\signals: 0


import sys
_missing = [v for v in ('SELECTED','all_contexts') if v not in dir()]
if _missing:
    print(f'⛔ Variáveis não definidas: {_missing}')
    print('   Execute a célula 13 (validação) antes de continuar.')
    sys.exit(0)

OVERWRITE = False   # True para forçar re-extracção
## 10. Visualização dos contextos

Plota o EEG de cada contexto válido mostrando as zonas coloridas.
Ajuste `MAX_CTX_PER_PAT` para limitar o número de plots por paciente.

In [13]:
def load_eeg_for_plot(edf_path, tmin_s, tmax_s):
    '''Carrega trecho sem filtros para visualizacao rapida (media de canais EEG).'''
    if not HAS_MNE or edf_path is None: return None, None
    try:
        raw = mne.io.read_raw_edf(str(edf_path), preload=False, verbose=False)
        sf  = raw.info['sfreq']
        tmin_s = max(0.0, float(tmin_s))
        tmax_s = min((raw.n_times - 1) / sf, float(tmax_s))
        if tmax_s <= tmin_s: raw.close(); return None, None
        non_eeg = {'spo2','hr','ekg','ecg','emg','eog','mk','status','annot','trig','stim'}
        eeg_idx = [i for i,ch in enumerate(raw.ch_names)
                   if ch.upper().startswith('EEG')] or                   [i for i,ch in enumerate(raw.ch_names)
                   if not any(kw in ch.lower() for kw in non_eeg)]
        data, _ = raw[eeg_idx or list(range(len(raw.ch_names))),
                      int(tmin_s*sf):int(tmax_s*sf)]
        raw.close()
        if data.size == 0: return None, None
        mv = np.max(np.abs(data))
        if mv == 0: return None, None
        if mv < 0.001: data *= 1e6
        elif mv < 1.0: data *= 1e3
        t = np.arange(data.shape[1]) / sf / 60.0
        return t, data.mean(axis=0)
    except Exception as e:
        print(f'    Erro plot {Path(str(edf_path)).name}: {e}')
        return None, None

def plot_context(dataset, pat, ctx, ctx_id):
    is_ded = dataset in ('CHBMIT',)
    on, off = ctx['on_s'], ctx['off_s']
    pre_ini = on - MAX_PRE_S
    guard_ini = pre_ini - GUARD_S
    i_ini, i_fim = ctx['inter_inicio_s'], ctx['inter_fim_s']
    inter_dur = i_fim - i_ini

    # Aviso se onset > duracao real
    if ctx['path'] is not None and HAS_MNE:
        try:
            _r = mne.io.read_raw_edf(str(ctx['path']), preload=False, verbose=False)
            _dur = _r.n_times / _r.info['sfreq']
            _r.close()
            if on > _dur:
                print(f'  AVISO: onset={m(on):.1f}min > duracao real ({_dur/60:.1f}min)')
        except: pass

    print(f'\n' + '='*62)
    print(f'  {dataset}/{pat}  |  Contexto #{ctx_id}  |  {ctx["edf"]}  crise #{ctx["sz_idx"]}')
    print('='*62)
    print(f'  Interictal : {m(inter_dur):.1f}min  [{m(i_ini):.1f}-{m(i_fim):.1f}min]  '
          f'fonte={ctx["inter_fonte"]}  arq={ctx["inter_arquivo"]}')
    print(f'  Pre-ictal  : {m(MAX_PRE_S):.0f}min  [{m(pre_ini):.1f}-{m(on):.1f}min]  '
          f'max_disp={m(ctx["pre_max_disp_s"]):.1f}min')
    print(f'  Ictal      : {m(off-on):.1f}min  onset={m(on):.1f}min  offset={m(off):.1f}min')

    def span(ax, a, b, ws, cor, al, lbl):
        am = (a - ws)/60; bm = (b - ws)/60
        if bm > 0: ax.axvspan(max(am,0), bm, color=cor, alpha=al, label=lbl)

    if is_ded:
        fig, (ax1, ax2) = plt.subplots(2,1, figsize=(14,5),
                                        gridspec_kw={'hspace':0.5})
        t_i, sig_i = load_eeg_for_plot(ctx['inter_path'], i_ini, i_fim)
        if t_i is not None:
            ax1.plot(t_i, sig_i, color='#1D9E75', lw=0.5)
            ax1.set_xlim(t_i[0], t_i[-1])
        else:
            ax1.text(0.5,0.5,'Sinal nao carregado',ha='center',va='center',
                     transform=ax1.transAxes,color='gray')
        ax1.set_title(f'INTERICTAL — {ctx["inter_arquivo"]}  [{m(i_ini):.0f}-{m(i_fim):.0f}min]',
                      fontsize=9)
        ax1.set_xlabel('Tempo (min)'); ax1.set_ylabel('uV')

        ws = max(0.0, guard_ini - 120.0)
        we = off + 120.0
        t_c, sig_c = load_eeg_for_plot(ctx['path'], ws, we)
        if t_c is not None:
            ax2.plot(t_c, sig_c, color='#333', lw=0.5)
            ax2.set_xlim(t_c[0], t_c[-1])
        else:
            ax2.text(0.5,0.5,'Sinal nao carregado',ha='center',va='center',
                     transform=ax2.transAxes,color='gray')
        span(ax2, guard_ini, pre_ini, ws, '#999', 0.20, f'guard {int(GUARD_S//60)}min')
        span(ax2, pre_ini,   on,      ws, '#534AB7', 0.25, f'pre-ictal {int(MAX_PRE_S//60)}min')
        span(ax2, on,        off,     ws, '#D85A30', 0.50, 'ictal')
        ax2.legend(fontsize=7, loc='upper left', framealpha=0.8)
        ax2.set_title(f'CRISE — {ctx["edf"]}  onset={m(on):.1f}min', fontsize=9)
        ax2.set_xlabel('Tempo (min)'); ax2.set_ylabel('uV')
    else:
        ws = max(0.0, min(guard_ini, i_ini) - 60.0)
        we = max(off, i_fim) + 120.0
        fig, ax = plt.subplots(1,1, figsize=(14,4))
        t_c, sig_c = load_eeg_for_plot(ctx['path'], ws, we)
        if t_c is not None:
            ax.plot(t_c, sig_c, color='#333', lw=0.5)
            ax.set_xlim(t_c[0], t_c[-1])
        else:
            ax.text(0.5,0.5,'Sinal nao carregado',ha='center',va='center',
                    transform=ax.transAxes,color='gray')
        span(ax, i_ini,    i_fim,   ws, '#1D9E75', 0.18, f'inter {int(MIN_INTER_S//60)}min')
        span(ax, guard_ini, pre_ini, ws, '#999', 0.18, f'guard {int(GUARD_S//60)}min')
        span(ax, pre_ini,   on,      ws, '#534AB7', 0.22, f'pre-ictal {int(MAX_PRE_S//60)}min')
        span(ax, on,        off,     ws, '#D85A30', 0.40, 'ictal')
        ax.legend(fontsize=7, loc='upper left', framealpha=0.8)
        ax.set_title(f'{dataset}/{pat} | ctx#{ctx_id} | {ctx["edf"]} crise#{ctx["sz_idx"]} '
                     f'({ctx["inter_fonte"]})', fontsize=9)
        ax.set_xlabel('Tempo (min)'); ax.set_ylabel('uV')

    plt.tight_layout(); plt.show()

print('Funções de plot definidas.')

Funções de plot definidas.


In [14]:
# ── Plot de todos os contextos válidos ────────────────────────────────────────
DATASETS_TO_PLOT = ['CHBMIT','Siena','SeizeIT2','Mendeley']
MAX_CTX_PER_PAT  = None  # None = todos; ex: 2 para testar rapido

for dataset in DATASETS_TO_PLOT:
    for pat in SELECTED[dataset]:
        if pat not in all_contexts.get(dataset, {}): continue
        ctxs_val = [c for c in all_contexts[dataset][pat] if c['valid']]
        if not ctxs_val: continue
        print(f'\n####### {dataset}/{pat} — {len(ctxs_val)} contextos #######')
        to_plot = ctxs_val if MAX_CTX_PER_PAT is None else ctxs_val[:MAX_CTX_PER_PAT]
        for k, ctx in enumerate(to_plot, 1):
            plot_context(dataset, pat, ctx, k)

## 11. Exportação do mapa de contextos para o NB2

Salva `data/mapa_contextos_nb1.json` com os timestamps exatos de cada
contexto válido. O NB2 lê esse arquivo para estimar a janela pré-ictal
individual por paciente (PRE_SEC) via PELT + K-Means.

In [15]:
export = []
for dataset, pats in SELECTED.items():
    for pat in pats:
        if pat not in all_contexts.get(dataset, {}): continue
        ctx_id = 0
        for c in all_contexts[dataset][pat]:
            if not c['valid']: continue
            ctx_id += 1
            # janelas fixas disponiveis: 10, 20, 30, 40min
            janelas = [w*60 for w in (10, 20, 30, 40) if w*60 <= c['pre_max_disp_s']]
            export.append({
                'dataset':       dataset,
                'paciente':      pat,
                'contexto_id':   ctx_id,
                'npz_path':      str(SIGNAL_DIR / f'{dataset}__{pat}__ctx{ctx_id:03d}.npz'),
                'arquivo_crise': c['edf'],
                'onset_s':       c['on_s'],
                'offset_s':      c['off_s'],
                'preictal': {
                    'arquivo':   c['edf'],
                    'inicio_s':  c['pre_inicio_s'],
                    'fim_s':     c['pre_fim_s'],
                    'max_disp_s': c['pre_max_disp_s'],
                    'janelas_fixas_s': janelas,
                },
                'interictal': {
                    'arquivo':   c['inter_arquivo'],
                    'inicio_s':  c['inter_inicio_s'],
                    'fim_s':     c['inter_fim_s'],
                    'duracao_s': c['inter_fim_s'] - c['inter_inicio_s'],
                    'fonte':     c['inter_fonte'],
                },
            })

out_path = ROOT / 'mapa_contextos_nb1.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(export, f, indent=2, ensure_ascii=False)

print(f'Exportados {len(export)} contextos para: {out_path}')
from collections import Counter
for ds, n in Counter(e['dataset'] for e in export).items():
    n_pat = len(set(e['paciente'] for e in export if e['dataset']==ds))
    print(f'  {ds}: {n} contextos em {n_pat} pacientes')

# Download (Google Colab)
try:
    from google.colab import files
    files.download(str(out_path))
except ImportError:
    print(f'Arquivo salvo em: {out_path.resolve()}')

Exportados 0 contextos para: data\mapa_contextos_nb1.json
Arquivo salvo em: D:\TCC\oldnoteboooks\data\mapa_contextos_nb1.json
